## Classification Model without Tuning

**Classification Model without Tuning**

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

data = pd.read_csv("Data/train.csv")
data
data.fillna(0, inplace = True)
data = pd.get_dummies(data, columns=["Sex", "Embarked"], drop_first=True)
X = data.drop(["Survived","Name","PassengerId","Ticket","Cabin"], axis = 1)
y = data["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
gb_model = GradientBoostingClassifier()
gb_model.fit(X_train, y_train)
y_pred = gb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy:{accuracy:.2f}")

Accuracy:0.80


**1. Hyperparameter Tuning using GridSeachCV**

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

data = pd.read_csv("Data/train.csv")
data.fillna(0, inplace=True)
data = pd.get_dummies(data, columns=["Sex", "Embarked"], drop_first=True)
X = data.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis = 1);
y = data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
param_grid = {
    'n_estimators':[50, 100, 200],
    'learning_rate':[0.01, 0.1, 0.2],
    'max_depth':[3,5,7]
}
gb_model = GradientBoostingClassifier()
grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print("Best Parameters:", best_params)
print(f"Best Model Accuracy:{accuracy_best}")

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best Model Accuracy:0.8044692737430168


**2. Hyperparameter Tuning using RandomizedSearchCV**

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

titanic_data = pd.read_csv("Data/train.csv")

titanic_data.fillna(0, inplace=True)
titanic_data = pd.get_dummies(titanic_data, columns=['Sex', 'Embarked'], drop_first=True)

X = titanic_data.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y = titanic_data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_dist = {
    'learning_rate': np.arange(0.01, 0.2, 0.01), 
    'n_estimators': [100, 200, 300, 400],  
    'max_depth': [3, 5, 7, 9],  
}

gb_model = GradientBoostingClassifier()

random_search = RandomizedSearchCV(estimator=gb_model, param_distributions=param_dist, n_iter=10, cv=5, scoring='accuracy', n_jobs=-1)

random_search.fit(X_train, y_train)

best_params = random_search.best_params_
best_model = random_search.best_estimator_

y_pred_best = best_model.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred_best)

print("Best Parameters:", best_params)
print(f"Best Model Accuracy: {accuracy_best}")

Best Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01}
Best Model Accuracy: 0.7932960893854749


**3. Hyperparameter Tuning using Optuna**

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import optuna

titanic_data = pd.read_csv("Data/train.csv")

titanic_data.fillna(0, inplace=True)
titanic_data = pd.get_dummies(titanic_data, columns=['Sex', 'Embarked'], drop_first=True)

X = titanic_data.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y = titanic_data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    param_space = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
    }

    gb_model = GradientBoostingClassifier(**param_space, validation_fraction=0.1, n_iter_no_change=5, random_state=42)
    gb_model.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return 1.0 - accuracy 

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

best_params_optuna = study.best_params
best_model_optuna = GradientBoostingClassifier(**best_params_optuna, validation_fraction=0.1, n_iter_no_change=5, random_state=42)
best_model_optuna.fit(X_train, y_train)

y_pred_best_optuna = best_model_optuna.predict(X_test)

accuracy_best_optuna = accuracy_score(y_test, y_pred_best_optuna)
print(f"Best Model Accuracy (Optuna): {accuracy_best_optuna}")

[I 2026-01-24 17:06:56,247] A new study created in memory with name: no-name-42eef5da-deaa-4e26-ba58-12a1f970bdfc
[I 2026-01-24 17:06:56,396] Trial 0 finished with value: 0.2122905027932961 and parameters: {'n_estimators': 200, 'learning_rate': 0.06716203098711669, 'max_depth': 5}. Best is trial 0 with value: 0.2122905027932961.
[I 2026-01-24 17:06:56,490] Trial 1 finished with value: 0.1955307262569832 and parameters: {'n_estimators': 200, 'learning_rate': 0.08698035019280312, 'max_depth': 5}. Best is trial 1 with value: 0.1955307262569832.
[I 2026-01-24 17:06:56,631] Trial 2 finished with value: 0.2011173184357542 and parameters: {'n_estimators': 200, 'learning_rate': 0.1718587550928812, 'max_depth': 7}. Best is trial 1 with value: 0.1955307262569832.
[I 2026-01-24 17:06:56,679] Trial 3 finished with value: 0.2122905027932961 and parameters: {'n_estimators': 50, 'learning_rate': 0.17345054045555686, 'max_depth': 3}. Best is trial 1 with value: 0.1955307262569832.
[I 2026-01-24 17:06:

Best Model Accuracy (Optuna): 0.8324022346368715
